In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel
from scipy.ndimage import gaussian_filter
import mne
from scipy import signal
from joblib import Parallel, delayed
plt.ioff()

In [ ]:
# set directories
bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"

deriv_dir_ffr = os.path.join(bids_root, "derivatives", "epochs-python-stimtrack")
deriv_dir_erp = os.path.join(bids_root, "derivatives", "cortical_filtered")

# draw contour plot around stat significant regions on t-statistic plot
ffr_contour = False
erp_contour = True

### Calculate Phase Consistency

Phase Consistency Analysis for Frequency-Following Responses (FFR)
Converted from MATLAB (Krizman & Nicol, FFR Workshop 2024, Chicago) with assistance from Claude Code

In [ ]:
"""
Minimal phase consistency function for FFR epochs.

Usage:
    phasecon, xaxis, yaxis, numsweeps = compute_phase_consistency_minimal(
        epochs_A,
        epochs_B,
        chunksize=0.04,
        overlap=0.036
    )
    
Just provide epochs for A and B polarities. The function automatically computes 
phase consistency for all 4 polarities: A, B, add=(A+B)/2, and sub=(A-B)/2.
"""


def compute_phase_consistency_minimal(
    epochs_A,
    epochs_B,
    chunksize=0.04,
    overlap=0.036,
    freqcap=2000
):
    """
    Compute phase consistency from FFR epochs.
    
    The function computes phase consistency for:
    - Polarity A (from epochs_A)
    - Polarity B (from epochs_B)  
    - ADD polarity: (A + B) / 2 (computed from phase vectors)
    - SUB polarity: (A - B) / 2 (computed from phase vectors)
    
    Parameters
    ----------
    epochs_A : mne.Epochs
        Individual epochs for polarity A
    epochs_B : mne.Epochs
        Individual epochs for polarity B
    chunksize : float
        Analysis window size in seconds (default: 0.04 = 40 ms)
    overlap : float
        Window overlap in seconds (default: 0.036 = 36 ms, gives ~4 ms step)
    freqcap : int
        Maximum frequency in Hz (default: 2000)
        
    Returns
    -------
    phasecon : dict
        Dictionary with keys 'A', 'B', 'add', 'sub' and phase consistency arrays (freq x time) as values
    xaxis : ndarray
        Time axis in milliseconds
    yaxis : ndarray
        Frequency axis in Hz
    numsweeps : int
        Number of sweeps used (minimum of A and B)
    
    Examples
    --------
    >>> phasecon, xaxis, yaxis, numsweeps = compute_phase_consistency_minimal(
    ...     epochs_A, epochs_B, chunksize=0.04, overlap=0.036
    ... )
    >>> 
    >>> # Plot
    >>> plot_phase_consistency(phasecon, xaxis, yaxis)
    """
    
    # Get sampling info from epochs
    sfreq = epochs_A.info['sfreq']
    tmin = epochs_A.times[0]
    tmax = epochs_A.times[-1]
    
    # Convert parameters to samples
    chunksizepts = int(np.round(chunksize * sfreq))
    overlappts = int(np.round(overlap * sfreq))
    
    # Create Hann window
    ramp = signal.windows.hann(chunksizepts)
    
    # Get epoch data for A and B
    data_A = epochs_A.get_data(picks='eeg')
    data_B = epochs_B.get_data(picks='eeg')
    
    # Average across channels if multiple
    if data_A.shape[1] > 1:
        print(f"Averaging across {data_A.shape[1]} channels")
        data_A = np.mean(data_A, axis=1)  # Shape: (n_epochs, n_times)
        data_B = np.mean(data_B, axis=1)
    else:
        data_A = data_A[:, 0, :]
        data_B = data_B[:, 0, :]
    
    # Determine minimum number of sweeps
    numsweeps = min(len(epochs_A), len(epochs_B))
    data_A = data_A[:numsweeps, :]
    data_B = data_B[:numsweeps, :]
    
    n_epochs, n_samples = data_A.shape
    
    # Calculate sliding window positions
    segstart = np.arange(0, n_samples - chunksizepts, chunksizepts - overlappts)
    segstop = segstart + chunksizepts
    

    def process_window(s, data_A, data_B, segstart, segstop, ramp, sfreq, freqcap):
        # Extract segments from all epochs
        segment_A = data_A[:, segstart[s]:segstop[s]] 
        segment_B = data_B[:, segstart[s]:segstop[s]]
        
        # Detrend (baseline to 0)
        segment_A_detr = segment_A - segment_A.mean(axis=-1, keepdims=True)
        segment_B_detr = segment_B - segment_B.mean(axis=-1, keepdims=True)
        
        # Apply Hann window
        segment_A_windowed = segment_A_detr * ramp[None, :]
        segment_B_windowed = segment_B_detr * ramp[None, :]
        
        # Compute FFT with 1 Hz resolution
        fft_A = np.fft.rfft(segment_A_windowed, n=int(sfreq), axis=-1)[:, :freqcap + 1]
        fft_B = np.fft.rfft(segment_B_windowed, n=int(sfreq), axis=-1)[:, :freqcap + 1]
        
        # Keep only frequencies up to freqcap
        fft_A /= (np.abs(fft_A) + 1e-10)
        fft_B /= (np.abs(fft_B) + 1e-10)
        return fft_A.mean(axis=0), fft_B.mean(axis=0)
    
    results = Parallel(n_jobs=-1, backend='threading', verbose=5)(
        delayed(process_window)(
            s, data_A, data_B, segstart, segstop, ramp, sfreq, freqcap
            ) 
            for s in range(len(segstart))
            )

    # Unpack results
    phase_avg_A = np.zeros((freqcap + 1, len(segstart)), dtype=complex)
    phase_avg_B = np.zeros((freqcap + 1, len(segstart)), dtype=complex)
    fft_A_list, fft_B_list = zip(*results)
    phase_avg_A = np.array(fft_A_list).T
    phase_avg_B = np.array(fft_B_list).T

    # Compute ADD and SUB from the averaged phase vectors
    phase_avg_add = (phase_avg_A + phase_avg_B) / 2
    phase_avg_sub = (phase_avg_A - phase_avg_B) / 2
    
    # Phase consistency is the absolute value
    phasecon = {
        'A': np.abs(phase_avg_A),
        'B': np.abs(phase_avg_B),
        'add': np.abs(phase_avg_add),
        'sub': np.abs(phase_avg_sub)
    }
    
    # Create axes
    xaxis = 1000 * np.linspace(
        tmin + (chunksize / 2),
        tmax - (chunksize / 2),
        len(segstart)
    )
    yaxis = np.arange(0, freqcap + 1)
    
    print(f"\nPhase consistency computed!")
    print(f"  Polarities: A, B, add, sub")
    print(f"  Number of sweeps: {numsweeps}")
    print(f"  Time axis: {xaxis[0]:.1f} to {xaxis[-1]:.1f} ms ({len(xaxis)} points)")
    print(f"  Freq axis: {yaxis[0]} to {yaxis[-1]} Hz ({len(yaxis)} points)")
    
    return phasecon, xaxis, yaxis, numsweeps


def plot_phase_consistency(
    phasecon,
    xaxis,
    yaxis,
    pol_names=None,
    vmax=0.14,
    ylim=None,
    figsize=(12, 8),
    cmap='viridis'
):
    """
    Plot phase consistency matrices.
    
    Parameters
    ----------
    phasecon : dict
        Dictionary of phase consistency arrays
    xaxis : ndarray
        Time axis in ms
    yaxis : ndarray
        Frequency axis in Hz
    pol_names : list, optional
        Order of polarities to plot. If None, uses dict order
    vmax : float
        Maximum value for colormap (default: 0.14)
    ylim : tuple, optional
        Frequency limits (min_freq, max_freq)
    figsize : tuple
        Figure size (default: (12, 8))
    cmap : str
        Colormap name (default: 'viridis')
    
    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    
    if pol_names is None:
        pol_names = list(phasecon.keys())
    
    n_pols = len(pol_names)
    
    # Determine subplot layout
    if n_pols <= 2:
        nrows, ncols = 1, n_pols
    elif n_pols <= 4:
        nrows, ncols = 2, 2
    else:
        nrows = int(np.ceil(n_pols / 3))
        ncols = 3
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if n_pols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, pol_name in enumerate(pol_names):
        if pol_name not in phasecon:
            continue
        
        ax = axes[idx]
        
        im = ax.imshow(
            phasecon[pol_name],
            aspect='auto',
            origin='lower',
            extent=[xaxis[0], xaxis[-1], yaxis[0], yaxis[-1]],
            vmin=0,
            vmax=vmax,
            cmap=cmap
        )
        
        if ylim is not None:
            ax.set_ylim(ylim)
        
        ax.set_title(pol_name, fontsize=14, fontweight='bold')
        ax.set_xlabel('Time (ms)', fontsize=12)
        ax.set_ylabel('Frequency (Hz)', fontsize=12)
        
        plt.colorbar(im, ax=ax, label='Phase Consistency')
    
    # Hide extra subplots
    for idx in range(len(pol_names), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    return fig


def plot_phase_consistency_masked(
    phasecon,
    xaxis,
    yaxis,
    numsweeps,
    alpha=0.01,
    pol_names=None,
    vmax=0.14,
    ylim=None,
    figsize=(12, 8)
):
    """
    Plot phase consistency with significance masking.
    
    Parameters
    ----------
    phasecon : dict
        Dictionary of phase consistency arrays
    xaxis : ndarray
        Time axis in ms
    yaxis : ndarray
        Frequency axis in Hz
    numsweeps : int
        Number of sweeps used in the analysis
    alpha : float
        Significance level (default: 0.01)
    pol_names : list, optional
        Order of polarities to plot
    vmax : float
        Maximum value for colormap
    ylim : tuple, optional
        Frequency limits
    figsize : tuple
        Figure size
        
    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    
    if pol_names is None:
        pol_names = list(phasecon.keys())
    
    # Calculate cutoff (same for all polarities)
    cutoff = np.sqrt(-np.log(alpha) / numsweeps)
    print(f"Significance cutoff (α={alpha}, n={numsweeps}): {cutoff:.4f}")
    
    # Create masked version
    maskedphasecon = {}
    for pol_name in pol_names:
        if pol_name not in phasecon:
            continue
        
        # Mask
        masked = phasecon[pol_name].copy()
        masked[masked < cutoff] = 0
        maskedphasecon[pol_name] = masked
    
    # Create colormap with black for zero
    cmap = plt.cm.viridis.copy()
    cmap.set_under('black')
    
    n_pols = len(pol_names)
    
    # Determine subplot layout
    if n_pols <= 2:
        nrows, ncols = 1, n_pols
    elif n_pols <= 4:
        nrows, ncols = 2, 2
    else:
        nrows = int(np.ceil(n_pols / 3))
        ncols = 3
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if n_pols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, pol_name in enumerate(pol_names):
        if pol_name not in maskedphasecon:
            continue
        
        ax = axes[idx]
        
        im = ax.imshow(
            maskedphasecon[pol_name],
            aspect='auto',
            origin='lower',
            extent=[xaxis[0], xaxis[-1], yaxis[0], yaxis[-1]],
            vmin=cutoff,  # Values below are 'under' (black)
            vmax=vmax,
            cmap=cmap
        )
        
        if ylim is not None:
            ax.set_ylim(ylim)
        
        ax.set_title(f"{pol_name} (p < {alpha})", fontsize=14, fontweight='bold')
        ax.set_xlabel('Time (ms)', fontsize=12)
        ax.set_ylabel('Frequency (Hz)', fontsize=12)
        
        plt.colorbar(im, ax=ax, label='Phase Consistency')
    
    # Hide extra subplots
    for idx in range(len(pol_names), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    return fig


# Example usage
if __name__ == "__main__":
    """
    Example assuming you have:
    
    epochs_A = mne.Epochs(...)  # Individual trials for polarity A
    epochs_B = mne.Epochs(...)  # Individual trials for polarity B
    """
    
    print("Example usage:")
    print("""
    # Compute phase consistency - just need the two epoch objects!
    phasecon, xaxis, yaxis, numsweeps = compute_phase_consistency_minimal(
        epochs_A,
        epochs_B,
        chunksize=0.04,
        overlap=0.036,
        freqcap=2000
    )
    
    # Plot all polarities (A, B, add, sub)
    fig1 = plot_phase_consistency(
        phasecon, xaxis, yaxis,
        pol_names=['A', 'B', 'add', 'sub'],
        ylim=(0, 1200)
    )
    
    # Plot with significance masking
    fig2 = plot_phase_consistency_masked(
        phasecon, xaxis, yaxis, numsweeps,
        alpha=0.01,
        ylim=(0, 1200)
    )
    
    plt.show()
    """)

### Save results

In [ ]:
for task_label in ['passive', 'active']: # 'active', 
    eps = sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-{task_label}_run-all_event-stimtrack_epochs.fif')
                  for sub_num in range(2, 35)) # for sub_num in range(2, 35))]
    for ep in eps:
        try:
            sub_label = os.path.basename(ep[0]).split('_')[0]
            print(f"Processing subject {sub_label} for task {task_label}...")

            sub_epochs = mne.read_epochs(ep[0], verbose=False)

            print(f"  Loaded epochs: {len(sub_epochs['1'])} positive, {len(sub_epochs['2'])} negative")

            phasecon, xaxis, yaxis, numsweeps = compute_phase_consistency_minimal(
                sub_epochs['1'],
                sub_epochs['2'],
                chunksize=0.04,
                overlap=0.036,
                freqcap=1200
            )

            print(f"  Phase consistency computed for subject {sub_label} with {numsweeps} sweeps")

            out_dir = os.path.join(deriv_dir, 'phase-consistency-matlab', sub_label, 'eeg')
            os.makedirs(out_dir, exist_ok=True)

            out_path = os.path.join(out_dir, f'{sub_label}_task-{task_label}_desc-phasecon_timfreq.npz')

            print(f"Saving phase consistency results to {out_path}")
            np.savez(out_path,
                    phasecon=np.stack(list(phasecon.values())),
                    xaxis=xaxis,
                    yaxis=yaxis,
                    numsweeps=numsweeps)

            print(f"Saved phase consistency for subject {sub_label} successfully!\n")
        except Exception as e:
            print(f"Error occurred: {e}\n")
            pass

### Read in derivatives and get grand average phase consistency

In [ ]:
def get_phase_consistency_grand_avg(deriv_dir, task, pol_names):

    files = sorted(glob(os.path.join(
        deriv_dir, 'phase-consistency', 'sub-*', 'eeg',
        f'*_task-{task}_desc-phasecon*_timfreq.npz'
        )))

    stacks = []
    for f in files:
        d = np.load(f)
        stacks.append(d['phasecon'])   # shape: (4, freq, time)
    xaxis = d['xaxis']
    yaxis = d['yaxis']

    grand_avg = np.mean(np.stack(stacks), axis=0)  # (4, freq, time)
    phasecon_avg = {pol: grand_avg[i] for i, pol in enumerate(pol_names)}

    plot_extent = [xaxis[0], xaxis[-1], yaxis[0], yaxis[-1]]

    return phasecon_avg, plot_extent

### Pixel-wise paired t-test, FDR corrected

In [ ]:
def phase_con_ttest(deriv_dir, pol_names, print_stats=False):
    # Load matched pairs (subjects with both active and passive files)
    act_files = {
        os.path.basename(f).split('_')[0]: f
        for f in glob(os.path.join(deriv_dir, 'phase-consistency', '*', 'eeg',
                                '*_task-active_desc-phasecon*_timfreq.npz'))
    }
    pas_files = {
        os.path.basename(f).split('_')[0]: f
        for f in glob(os.path.join(deriv_dir, 'phase-consistency', '*', 'eeg',
                                '*_task-passive_desc-phasecon*_timfreq.npz'))
    }
    common_subs = sorted(set(act_files) & set(pas_files))
    print(f"Matched subjects: {len(common_subs)}")

    act_stack = np.stack([np.load(act_files[s])['phasecon'] for s in common_subs])  # (n_sub, 4, freq, time)
    pas_stack = np.stack([np.load(pas_files[s])['phasecon'] for s in common_subs])

    d = np.load(act_files[common_subs[0]])
    xaxis, yaxis = d['xaxis'], d['yaxis']

    extent = [xaxis[0], xaxis[-1], yaxis[0], yaxis[-1]]

    t_maps = {}

    if print_stats:
        print(f"\n{'Polarity':<12} {'Sig. pixels':<14} {'% of total':<12} {'Peak t':<10} {'Peak freq (Hz)':<16} {'Peak time (ms)'}")
        print('-' * 76)

    # Pixel-wise paired t-test for each polarity, FDR corrected
    for i, pol in enumerate(pol_names):
        t_map, p_map = ttest_rel(act_stack[:, i], pas_stack[:, i], axis=0)  # (freq, time)

        # FDR correction across all pixels
        reject, p_corr, _, _ = multipletests(p_map.ravel(), alpha=0.05, method='fdr_bh')
        sig_mask = reject.reshape(p_map.shape)

        t_maps[pol] = t_map

        if print_stats:
            n_sig = sig_mask.sum()
            pct = 100 * n_sig / sig_mask.size
            peak_idx = np.unravel_index(np.abs(t_map).argmax(), t_map.shape)
            peak_t = t_map[peak_idx]
            peak_freq = yaxis[peak_idx[0]]
            peak_time = xaxis[peak_idx[1]]
            print(f"{pol:<12} {n_sig:<14} {pct:<12.1f} {peak_t:<10.3f} {peak_freq:<16.0f} {peak_time:.1f}")
    
    return t_maps, extent, sig_mask

### FFR phase consistency

In [ ]:
pol_names = ['addition']

# passive phase consistency
passive_phasecon_ffr, passive_ffr_extent = get_phase_consistency_grand_avg(
    deriv_dir_ffr, 
    'passive', 
    pol_names)

# active phase consistency
active_phasecon_ffr, active_ffr_extent = get_phase_consistency_grand_avg(
    deriv_dir_ffr, 
    'active', 
    pol_names)

# t-statistic map
ffr_t_maps, ffr_t_extent, ffr_sig_mask = phase_con_ttest(deriv_dir_ffr, pol_names)

### ERP phase consistency

In [ ]:
pol_names = ['addition']

# passive phase consistency
passive_phasecon_erp, passive_erp_extent = get_phase_consistency_grand_avg(
    deriv_dir_erp, 
    'passive', 
    pol_names)

# active phase consistency
active_phasecon_erp, active_erp_extent = get_phase_consistency_grand_avg(
    deriv_dir_erp, 
    'active', 
    pol_names)

# t-statistic map
erp_t_maps, erp_t_extent, erp_sig_mask = phase_con_ttest(deriv_dir_erp, pol_names)

## Plot phase consistencies

Figure 4 in manuscript

In [ ]:
# set up subplot orientation
plt.close("all")
fig, axd = plt.subplot_mosaic(
    [
        ['ffr_passive', 'ffr_active', 'ffr_t_stat'],
        ['erp_passive', 'erp_active', 'erp_t_stat'],
        ['phase_con_cbar', 'phase_con_cbar', 't_stat_cbar']
    ],
    height_ratios=[1, 1, 0.2],
    figsize=(12, 10),
    constrained_layout=True
)

SMALL  = 13    # tick labels
MEDIUM = 14    # axis labels, colorbar label  
LARGE  = 15    # subplot titles

plt.rcParams.update({
    'font.size':        MEDIUM,
    'axes.titlesize':   LARGE,
    'axes.labelsize':   MEDIUM,
    'xtick.labelsize':  MEDIUM,
    'ytick.labelsize':  MEDIUM,
})

Plot FFR phase consistency

In [ ]:

# ffr phase consistency
im_ffr_p = axd['ffr_passive'].imshow(passive_phasecon_ffr['addition'],
                                     aspect='auto', 
                                     origin='lower',
                                     extent=passive_ffr_extent,
                                     vmin=0,
                                     vmax=0.14,
                                     cmap='magma')

im_ffr_a = axd['ffr_active'].imshow(active_phasecon_ffr['addition'], 
                                    aspect='auto', 
                                    origin='lower',
                                    extent=active_ffr_extent,
                                    vmin=0,
                                    vmax=0.14,
                                    cmap='magma')

# ffr t-statistic map
for pol in pol_names:
    im_ffr_t = axd["ffr_t_stat"].imshow(ffr_t_maps[pol],
                                        aspect='auto', 
                                        origin='lower',
                                        extent=ffr_t_extent,
                                        cmap='RdBu_r', 
                                        vmin=-4, 
                                        vmax=4)
    
    if ffr_contour:
        mask_smooth = gaussian_filter(ffr_sig_mask.astype(float), sigma=1)
        axd["ffr_t_stat"].contour(mask_smooth, 
                                  levels=[0.5],
                                  colors='black', 
                                  linewidths=2,
                                  extent=ffr_t_extent)
    
    # colorbar: ffr t-statistic map
    plt.draw()

    # formatting
    axd["ffr_active"].set_title("FFR: Active")
    axd["ffr_active"].set_xlabel("Time (ms)")
    axd["ffr_active"].set_ylabel("")
    axd['ffr_active'].tick_params(direction="in")

    axd["ffr_passive"].set_title("FFR: Passive")
    axd["ffr_passive"].set_xlabel("Time (ms)")
    axd["ffr_passive"].set_ylabel("Frequency (Hz)")
    axd["ffr_passive"].tick_params(direction="in")

    axd['ffr_t_stat'].set_title(f't-statistic (Active − Passive)')
    axd['ffr_t_stat'].set_ylim(0, 1200)
    axd['ffr_t_stat'].set_xlabel('Time (ms)')
    axd['ffr_t_stat'].tick_params(direction="in")

Plot ERP phase consistency

In [ ]:
# erp phase consistency
im_erp_p = axd['erp_passive'].imshow(passive_phasecon_erp['addition'],
                                     aspect='auto', 
                                     origin='lower',
                                     extent=passive_erp_extent,
                                     vmin=0,
                                     vmax=0.14,
                                     cmap='magma')

im_erp_a = axd['erp_active'].imshow(active_phasecon_erp['addition'], 
                                    aspect='auto', 
                                    origin='lower',
                                    extent=active_erp_extent,
                                    vmin=0,
                                    vmax=0.14,
                                    cmap='magma')


cb_im_erp = axd["erp_passive"].images[0]
pos = axd['phase_con_cbar'].get_position()
cax_phase = fig.add_axes([pos.x0 + pos.width * 0.05,   # left padding
                          pos.y0 + pos.height * 0.4,    # vertical centering
                          pos.width * 0.90,              # 90% of available width
                          pos.height * 0.2])
cb_erp_1 = plt.colorbar(cb_im_erp,
                        ax=[axd['phase_con_cbar']],
                        cax=cax_phase,
                        location='bottom',
                        label="Phase Consistency")

# erp t-statistic map
for pol in pol_names:
    im_ffr_t = axd["erp_t_stat"].imshow(erp_t_maps[pol],
                                        aspect='auto', 
                                        origin='lower',
                                        extent=erp_t_extent,
                                        cmap='RdBu_r', 
                                        vmin=-4, 
                                        vmax=4)
    
    if erp_contour:
          mask_smooth = gaussian_filter(erp_sig_mask.astype(float), sigma=1)
          axd["erp_t_stat"].contour(mask_smooth, 
                                    levels=[0.5],
                                    colors='black', 
                                    linewidths=2,
                                    extent=erp_t_extent)
    
    # colorbar: erp t-statistic map
    plt.draw()
    cb_im_ffr_t = axd["erp_t_stat"].images[0]
    pos = axd['t_stat_cbar'].get_position()
    cax_tstat = fig.add_axes([pos.x0 + pos.width * 0.05,   # left padding
                            pos.y0 + pos.height * 0.4,    # vertical centering
                            pos.width * 0.90,              # 90% of available width
                            pos.height * 0.2])
    
    cb_ffr_t = plt.colorbar(cb_im_ffr_t,
                        ax=axd["t_stat_cbar"],
                        cax=cax_tstat,
                        orientation='horizontal',
                        label='t-statistic')

# formatting
    axd["erp_active"].set_title("ERP: Active")
    axd["erp_active"].set_xlabel("Time (ms)")
    axd["erp_active"].set_ylabel("")
    axd["erp_active"].tick_params(direction="in")

    axd["erp_passive"].set_title("ERP: Passive")
    axd["erp_passive"].set_xlabel("Time (ms)")
    axd["erp_passive"].set_ylabel("Frequency (Hz)")
    axd["erp_passive"].tick_params(direction="in")

    axd['erp_t_stat'].set_title(f't-statistic (Active − Passive)')
    axd['erp_t_stat'].set_ylim(0, 30)
    axd['erp_t_stat'].set_xlabel('Time (ms)')
    axd['erp_t_stat'].tick_params(direction="in")

    axd['phase_con_cbar'].axis("off")
    axd['t_stat_cbar'].axis("off")

In [ ]:
# save figure
fig.savefig('phase_consistency.svg', dpi=600)